In [1]:
import os
import torch.nn as nn
import torch.optim as optim

import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms

from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset



SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=5),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.05
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.45, 0.45, 0.45),
        (0.225, 0.225, 0.225)
    )
])


test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.45, 0.45, 0.45),
        (0.225, 0.225, 0.225)
    )
])



data_path = "./data_images"

dataset = torchvision.datasets.ImageFolder(
    data_path,
    transform=train_transforms
)

print("Classes:", dataset.class_to_idx)
print("Total images:", len(dataset))

Classes: {'data_images': 0}
Total images: 200


In [2]:
indices = np.arange(len(dataset))
labels = np.array(dataset.targets)

train_indices, temp_indices = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)


val_indices, test_indices = train_test_split(
    temp_indices,
    test_size=0.50,
    random_state=SEED,
    stratify=labels[temp_indices]
)


print("Train:", len(train_indices))
print("Validation:", len(val_indices))
print("Test:", len(test_indices))

Train: 140
Validation: 30
Test: 30


In [3]:
train_dataset_full = torchvision.datasets.ImageFolder(
    data_path,
    transform=train_transforms
)

test_dataset_full = torchvision.datasets.ImageFolder(
    data_path,
    transform=test_transforms
)

In [4]:
train_subset = Subset(
    train_dataset_full,
    train_indices
)

val_subset = Subset(
    test_dataset_full,
    val_indices
)

test_subset = Subset(
    test_dataset_full,
    test_indices
)

In [5]:
trainloader = DataLoader(
    train_subset,
    batch_size=4,
    shuffle=True
)

valloader = DataLoader(
    val_subset,
    batch_size=4,
    shuffle=False
)

testloader = DataLoader(
    test_subset,
    batch_size=4,
    shuffle=False
)

In [6]:
images, labels = next(iter(trainloader))

print("Image shape:", images.shape)
print("Labels:", labels)

Image shape: torch.Size([4, 3, 224, 224])
Labels: tensor([0, 0, 0, 0])


In [7]:

class ImageCNN(nn.Module):

    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(

            # Convolutional layer 1
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Convolutional layer 2
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Convolutional layer 3
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)

        x = x.view(x.size(0), -1)

        x = self.classifier(x)

        return x


model = ImageCNN(num_classes=2)

print(model)

ImageCNN(
  (features): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=50176, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)


In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score


model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():

    for images, labels in testloader:

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_predictions.extend(predicted.numpy())
        all_labels.extend(labels.numpy())


# Calculate metrics

test_accuracy = accuracy_score(
    all_labels,
    all_predictions
)

test_precision = precision_score(
    all_labels,
    all_predictions,
    average="binary"
)

test_recall = recall_score(
    all_labels,
    all_predictions,
    average="binary"
)


print(f"Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")

Test Accuracy: 80.00%
Test Precision: 0.0000
Test Recall: 0.0000


c:\Users\badet\anaconda\envs\mesa-env\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [9]:
# Run this in your notebook after training
torch.save(model.state_dict(), "image_cnn_weights.pth")
print("Weights saved successfully to image_cnn_weights.pth!")

Weights saved successfully to image_cnn_weights.pth!
